In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.initializers import HeNormal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def set_seed(seed=42):

    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(42)

In [ ]:
X = pd.read_csv("/content/drive/MyDrive/Final Examination Deep Learning/full-data-input_updated.csv")
y = pd.read_csv("/content/drive/MyDrive/Final Examination Deep Learning/full-data-output_updated_STI.csv")

print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (10816, 89)
Output shape: (10816, 3)


In [ ]:
X = X.values
y = y.values

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
X_train = scaler_X.fit_transform(X_train)
X_val   = scaler_X.transform(X_val)
X_test  = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_val   = scaler_y.transform(y_val)
y_test  = scaler_y.transform(y_test)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (8652, 89) (8652, 3)
Val: (1082, 89) (1082, 3)
Test: (1082, 89) (1082, 3)


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
def split_direct_ray(X_array):
    # Direct features: [LSR, Sx, Sy, Rx, Ry]
    direct_all = np.concatenate([
        X_array[:, 28:29],   # out:LSR
        X_array[:, 85:89]    # out:Sx, out:Sy, out:Rx, out:Ry
    ], axis=1)

    # Ray features grouped correctly per ray:
    # each ray = [LiS, LiR, ri, Pix, Piy, Piz]
    Ls  = X_array[:, 0:14]    # out:L01S ... out:L14S
    Lr  = X_array[:, 14:28]   # out:L01R ... out:L14R
    r   = X_array[:, 29:43]   # out:r01  ... out:r14
    Px  = X_array[:, 43:57]   # out:P01x ... out:P14x
    Py  = X_array[:, 57:71]   # out:P01y ... out:P14y
    Pz  = X_array[:, 71:85]   # out:P01z ... out:P14z

    ray_all = np.stack([Ls, Lr, r, Px, Py, Pz], axis=2)   # shape: (N, 14, 6)

    return direct_all, ray_all

In [ ]:
direct_all, ray_all = split_direct_ray(X)

print("direct_all shape:", direct_all.shape)
print("ray_all shape:", ray_all.shape)

print("direct sample:", direct_all[0])
print("ray 1 sample:", ray_all[0, 0, :])
print("ray 14 sample:", ray_all[0, 13, :])

direct_all shape: (10816, 5)
ray_all shape: (10816, 14, 6)
direct sample: [0.4  0.25 0.25 0.25 0.25]
ray 1 sample: [  3.95       3.970202 174.217608   0.25       4.2        1.5     ]
ray 14 sample: [  0.433013   0.384057 121.745843   0.         0.         1.25    ]


In [ ]:
direct_train, ray_train = split_direct_ray(X_train)
direct_val, ray_val = split_direct_ray(X_val)
direct_test, ray_test = split_direct_ray(X_test)

In [ ]:
direct_train_t = torch.tensor(direct_train, dtype=torch.float32)
direct_val_t   = torch.tensor(direct_val, dtype=torch.float32)
direct_test_t  = torch.tensor(direct_test, dtype=torch.float32)

ray_train_t = torch.tensor(ray_train, dtype=torch.float32)
ray_val_t   = torch.tensor(ray_val, dtype=torch.float32)
ray_test_t  = torch.tensor(ray_test, dtype=torch.float32)

In [ ]:
batch_size = 128

ray_train_loader = DataLoader(
    TensorDataset(direct_train_t, ray_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True
)

ray_val_loader = DataLoader(
    TensorDataset(direct_val_t, ray_val_t, y_val_t),
    batch_size=batch_size,
    shuffle=False
)

ray_test_loader = DataLoader(
    TensorDataset(direct_test_t, ray_test_t, y_test_t),
    batch_size=batch_size,
    shuffle=False
)

print("Direct train shape:", direct_train.shape)
print("Ray train shape:", ray_train.shape)

Direct train shape: (8652, 5)
Ray train shape: (8652, 14, 6)


Initialize parameter by He Normalization

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

Structured ray-based model

In [ ]:
class DirectEncoder(nn.Module):
    def __init__(self, input_dim=5, emb_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, emb_dim),
            nn.ReLU()
        )
        self.net.apply(init_weights)

    def forward(self, x):
        return self.net(x)

In [ ]:
class RayEncoder(nn.Module):
    def __init__(self, input_dim=6, emb_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, emb_dim),
            nn.ReLU()
        )
        self.net.apply(init_weights)

    def forward(self, x):
        return self.net(x)

TwoBranch ray model

In [ ]:
class TwoBranchRayNet(nn.Module):
    def __init__(self, direct_dim=5, ray_dim=6, ray_emb_dim=32, output_dim=15):
        super().__init__()

        self.direct_encoder = DirectEncoder(input_dim=direct_dim, emb_dim=32)
        self.ray_encoder = RayEncoder(input_dim=ray_dim, emb_dim=ray_emb_dim)

        self.fusion = nn.Sequential(

            nn.Linear(32 + ray_emb_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.15),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.15),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.15),

            nn.Linear(128, 64),
            nn.GELU(),

            nn.Linear(64, output_dim)
        )
        self.fusion.apply(init_weights)

    def forward(self, direct, ray):
        B = ray.size(0)

        direct_emb = self.direct_encoder(direct)

        ray_flat = ray.view(-1, ray.size(-1))
        ray_emb = self.ray_encoder(ray_flat)
        ray_emb = ray_emb.view(B, 14, -1)

        ray_global = ray_emb.mean(dim=1)

        fused = torch.cat([direct_emb, ray_global], dim=1)
        return self.fusion(fused)

Initialize model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

two_branch_model = TwoBranchRayNet(direct_dim=5, ray_dim=6, ray_emb_dim=32, output_dim=3).to(device)

Using device: cuda


In [ ]:
class EarlyStopping:
    def __init__(self, patience=50, min_delta=1e-5):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

In [ ]:
def train_ray_model(model, train_loader, val_loader, epochs=1000, lr=5e-4):
    model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=20, factor=0.5
    )
    early_stopping = EarlyStopping(patience=60)

    train_losses = []
    val_losses = []
    best_state = None
    best_val = float("inf")

    for epoch in range(epochs):
        model.train()
        running_train = 0.0

        for direct_b, ray_b, y_b in train_loader:
            direct_b = direct_b.to(device)
            ray_b = ray_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()
            pred = model(direct_b, ray_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()

            running_train += loss.item() * direct_b.size(0)

        train_loss = running_train / len(train_loader.dataset)

        model.eval()
        running_val = 0.0
        with torch.no_grad():
            for direct_b, ray_b, y_b in val_loader:
                direct_b = direct_b.to(device)
                ray_b = ray_b.to(device)
                y_b = y_b.to(device)

                pred = model(direct_b, ray_b)
                loss = criterion(pred, y_b)
                running_val += loss.item() * direct_b.size(0)

        val_loss = running_val / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()

        early_stopping.step(val_loss)

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train: {train_loss:.6f} - Val: {val_loss:.6f}")

        if early_stopping.should_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses

In [ ]:
print("\n===== Training TwoBranchRayNet =====")
two_branch_model, ray_train_losses, ray_val_losses = train_ray_model(
    two_branch_model, ray_train_loader, ray_val_loader, epochs=1000, lr=5e-4
)



===== Training TwoBranchRayNet =====
Epoch 20/1000 - Train: 0.045785 - Val: 0.022007
Epoch 40/1000 - Train: 0.029942 - Val: 0.018381
Epoch 60/1000 - Train: 0.024696 - Val: 0.018605
Epoch 80/1000 - Train: 0.021665 - Val: 0.016638
Epoch 100/1000 - Train: 0.019738 - Val: 0.015357
Epoch 120/1000 - Train: 0.017270 - Val: 0.015684
Epoch 140/1000 - Train: 0.016259 - Val: 0.015116
Epoch 160/1000 - Train: 0.014572 - Val: 0.013906
Epoch 180/1000 - Train: 0.013945 - Val: 0.014256
Epoch 200/1000 - Train: 0.013491 - Val: 0.013517
Early stopping at epoch 217


In [ ]:
torch.save(two_branch_model.state_dict(), "model.pth")
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")

['scaler_y.pkl']

In [ ]:
from google.colab import files
files.download("model.pth")
files.download("scaler_X.pkl")
files.download("scaler_y.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Evaluation

In [ ]:
def mape(y_true, y_pred):
    mask = y_true !=0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [ ]:
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / (denom + 1e-8)) * 100

In [ ]:
def evaluate_ray_model(model, test_loader, scaler_y):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for direct_b, ray_b, y_b in test_loader:
            direct_b = direct_b.to(device)
            ray_b = ray_b.to(device)

            pred = model(direct_b, ray_b).cpu().numpy()
            preds.append(pred)
            trues.append(y_b.numpy())

    preds = np.vstack(preds)
    trues = np.vstack(trues)

    preds_inv = scaler_y.inverse_transform(preds)
    trues_inv = scaler_y.inverse_transform(trues)

    mae = mean_absolute_error(trues_inv, preds_inv)
    r2 = r2_score(trues_inv, preds_inv)
    rmse = np.sqrt(mean_squared_error(trues_inv, preds_inv))
    m_ape = mape(trues_inv, preds_inv)
    s_mape = smape(trues_inv, preds_inv)

    return preds_inv, trues_inv, mae, r2, rmse, m_ape, s_mape

In [ ]:
def compute_metrics_per_output(y_true, y_pred):
    mae_list, rmse_list, mape_list, smape_list, r2_list = [], [], [], [], []

    for i in range(y_true.shape[1]):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        mae_i = mean_absolute_error(yt, yp)
        rmse_i = np.sqrt(mean_squared_error(yt, yp))

        mask = yt != 0
        mape_i = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.sum() > 0 else 0.0

        denom = (np.abs(yt) + np.abs(yp)) / 2.0
        smape_i = np.mean(np.abs(yt - yp) / (denom + 1e-8)) * 100

        r2_i = r2_score(yt, yp)

        mae_list.append(mae_i)
        rmse_list.append(rmse_i)
        mape_list.append(mape_i)
        smape_list.append(smape_i)
        r2_list.append(r2_i)

    return {
        "MAE": mae_list,
        "RMSE": rmse_list,
        "MAPE": mape_list,
        "sMAPE": smape_list,
        "R2": r2_list
    }

In [ ]:
pred_real_ray, y_real_ray, mae_ray, r2_ray, rmse_ray, mape_ray, smape_ray = evaluate_ray_model(
    two_branch_model, ray_test_loader, scaler_y
)

val_pred_ray, val_true_ray, val_mae_ray, val_r2_ray, val_rmse_ray, val_mape_ray, val_smape_ray = evaluate_ray_model(
    two_branch_model, ray_val_loader, scaler_y
)


print("\n==============================")
print("TwoBranchRayNet - VALIDATION")
print("==============================")
print(f"MAE   : {val_mae_ray:.6f}")
print(f"R2    : {val_r2_ray:.6f}")
print(f"RMSE  : {val_rmse_ray:.6f}")
print(f"MAPE  : {val_mape_ray:.6f}")
print(f"sMAPE : {val_smape_ray:.6f}")

print("\n==============================")
print("TwoBranchRayNet - TEST")
print("==============================")
print(f"MAE   : {mae_ray:.6f}")
print(f"R2    : {r2_ray:.6f}")
print(f"RMSE  : {rmse_ray:.6f}")
print(f"MAPE  : {mape_ray:.6f}")
print(f"sMAPE : {smape_ray:.6f}")



TwoBranchRayNet - VALIDATION
MAE   : 0.003542
R2    : 0.985650
RMSE  : 0.004586
MAPE  : 0.553128
sMAPE : 0.552982

TwoBranchRayNet - TEST
MAE   : 0.003588
R2    : 0.985564
RMSE  : 0.004543
MAPE  : 0.559183
sMAPE : 0.559039


In [ ]:
test_pred_ray, test_true_ray = pred_real_ray, y_real_ray

metrics_val_ray = compute_metrics_per_output(val_true_ray, val_pred_ray)
metrics_test_ray = compute_metrics_per_output(test_true_ray, test_pred_ray)

In [ ]:
def print_metrics_per_sti(metrics_val, metrics_test, model_name, output_names):
    print(f"\n{'='*60}")
    print(f"{model_name} - CHI TIẾT METRIC THEO TỪNG CHỈ SỐ STI")
    print(f"{'='*60}")

    for i, out_name in enumerate(output_names):
        print(f"\n--- {out_name} ---")
        print("Validation:")
        print(f"  MAE   : {metrics_val['MAE'][i]:.6f}")
        print(f"  RMSE  : {metrics_val['RMSE'][i]:.6f}")
        print(f"  MAPE  : {metrics_val['MAPE'][i]:.6f}")
        print(f"  sMAPE : {metrics_val['sMAPE'][i]:.6f}")
        print(f"  R2    : {metrics_val['R2'][i]:.6f}")

        print("Test:")
        print(f"  MAE   : {metrics_test['MAE'][i]:.6f}")
        print(f"  RMSE  : {metrics_test['RMSE'][i]:.6f}")
        print(f"  MAPE  : {metrics_test['MAPE'][i]:.6f}")
        print(f"  sMAPE : {metrics_test['sMAPE'][i]:.6f}")
        print(f"  R2    : {metrics_test['R2'][i]:.6f}")


output_names = ["STI_1", "STI_2", "STI_3"]

print_metrics_per_sti(metrics_val_ray, metrics_test_ray, "TwoBranchRayNet", output_names)


TwoBranchRayNet - CHI TIẾT METRIC THEO TỪNG CHỈ SỐ STI

--- STI_1 ---
Validation:
  MAE   : 0.003646
  RMSE  : 0.004731
  MAPE  : 0.601446
  sMAPE : 0.601232
  R2    : 0.987247
Test:
  MAE   : 0.003701
  RMSE  : 0.004664
  MAPE  : 0.609228
  sMAPE : 0.608987
  R2    : 0.987371

--- STI_2 ---
Validation:
  MAE   : 0.003462
  RMSE  : 0.004473
  MAPE  : 0.528164
  sMAPE : 0.528093
  R2    : 0.984875
Test:
  MAE   : 0.003499
  RMSE  : 0.004444
  MAPE  : 0.532608
  sMAPE : 0.532549
  R2    : 0.984690

--- STI_3 ---
Validation:
  MAE   : 0.003518
  RMSE  : 0.004550
  MAPE  : 0.529776
  sMAPE : 0.529620
  R2    : 0.984829
Test:
  MAE   : 0.003565
  RMSE  : 0.004520
  MAPE  : 0.535714
  sMAPE : 0.535581
  R2    : 0.984631


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# 1. Đọc lại file dữ liệu (giả sử bạn đã load df trước đó)
df = pd.read_csv("/content/drive/MyDrive/Final Examination Deep Learning/full-data-input_updated.csv")
# 2. Định nghĩa 5 biến đầu vào (Camera thu được)
direct_features = ['out:Sx', 'out:Sy', 'out:Rx', 'out:Ry', 'out:LSR']

# 3. Lấy 84 biến còn lại làm mục tiêu (Target) cho mô hình phụ
ray_features = [col for col in df.columns if col not in direct_features]

# Kiểm tra đảm bảo đủ 89 biến
print(f"Số lượng biến Direct (X): {len(direct_features)}")
print(f"Số lượng biến Ray (y): {len(ray_features)}")

# 4. Tách X và y
X_surrogate = df[direct_features].values
y_surrogate = df[ray_features].values

# 5. Chia tập train/test để đánh giá (80% train, 20% test)
X_sur_train, X_sur_test, y_sur_train, y_sur_test = train_test_split(
    X_surrogate, y_surrogate, test_size=0.2, random_state=42
)

Số lượng biến Direct (X): 5
Số lượng biến Ray (y): 84


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error
import joblib
# ==========================================
# 2. FEATURE ENGINEERING (TẠO THÊM ĐẶC TRƯNG)
# ==========================================
def enhance_features(X_data):
    # X_data ban đầu có 5 cột: Sx(0), Sy(1), Rx(2), Ry(3), LSR(4)
    Sx, Sy = X_data[:, 0], X_data[:, 1]
    Rx, Ry = X_data[:, 2], X_data[:, 3]

    # Tính thêm các biến hình học để "nhắc bài" cho mô hình
    dx = np.abs(Sx - Rx)
    dy = np.abs(Sy - Ry)
    area = dx * dy
    sum_xy = Sx + Sy + Rx + Ry

    # Ghép thêm vào X (Từ 5 biến mở rộng thành 9 biến)
    enhanced_X = np.column_stack((X_data, dx, dy, area, sum_xy))
    return enhanced_X

# Áp dụng cho cả tập train và test
X_train_enh = enhance_features(X_sur_train)
X_test_enh = enhance_features(X_sur_test)

# ==========================================
# 3. CẤU HÌNH XGBOOST NÂNG CAO
# ==========================================
xgb_advanced = xgb.XGBRegressor(
    n_estimators=1000,      # Tăng số cây lên 1000 để học thật kỹ
    learning_rate=0.01,     # Giảm tốc độ học để tối ưu chi tiết
    max_depth=9,            # Cho phép cây sâu hơn để bắt quy luật phức tạp
    subsample=0.8,          # Lấy mẫu ngẫu nhiên 80% data mỗi lần (chống quá khớp)
    colsample_bytree=0.8,   # Lấy ngẫu nhiên 80% đặc trưng mỗi lần (chống quá khớp)
    tree_method='hist',     # Dùng hist để train nhanh trên dữ liệu lớn
    random_state=42
)

xgb_multi_adv = MultiOutputRegressor(xgb_advanced, n_jobs=-1)

print(f"Đang huấn luyện XGBoost nâng cao với {X_train_enh.shape[1]} biến đầu vào...")
xgb_multi_adv.fit(X_train_enh, y_sur_train)
print("Huấn luyện xong!\n")

# ==========================================
# 4. DỰ ĐOÁN VÀ ĐÁNH GIÁ
# ==========================================
y_pred_adv = xgb_multi_adv.predict(X_test_enh)
mae_adv_total = mean_absolute_error(y_sur_test, y_pred_adv)
print(f"=====> MAE TỔNG TRÊN 84 BIẾN: {mae_adv_total:.4f} <=====\n")

# ==========================================
# 5. ĐÁNH GIÁ CHI TIẾT TỪNG NHÓM BIẾN
# ==========================================
group_dict = {
    'Distance_LS (L01S->L14S)': [i for i, col in enumerate(ray_features) if 'L' in col and 'S' in col],
    'Distance_LR (L01R->L14R)': [i for i, col in enumerate(ray_features) if 'L' in col and 'R' in col],
    'Reflect_Coef (r01->r14)': [i for i, col in enumerate(ray_features) if 'r0' in col or 'r1' in col],
    'Coord_Px (P01x->P14x)': [i for i, col in enumerate(ray_features) if 'Px' in col],
    'Coord_Py (P01y->P14y)': [i for i, col in enumerate(ray_features) if 'Py' in col],
    'Coord_Pz (P01z->P14z)': [i for i, col in enumerate(ray_features) if 'Pz' in col],
}

print("=== ĐÁNH GIÁ MAE CHI TIẾT TỪNG NHÓM ===")
for group_name, indices in group_dict.items():
    if len(indices) > 0:
        mae_group = mean_absolute_error(y_sur_test[:, indices], y_pred_adv[:, indices])
        print(f"Nhóm {group_name:<25} ({len(indices):>2} biến): MAE = {mae_group:.4f}")

Đang huấn luyện XGBoost nâng cao với 9 biến đầu vào...
Huấn luyện xong!

=====> MAE TỔNG TRÊN 84 BIẾN: 0.2042 <=====

=== ĐÁNH GIÁ MAE CHI TIẾT TỪNG NHÓM ===
Nhóm Distance_LS (L01S->L14S)  (14 biến): MAE = 0.0132
Nhóm Distance_LR (L01R->L14R)  (14 biến): MAE = 0.0334
Nhóm Reflect_Coef (r01->r14)   (14 biến): MAE = 1.1445


In [ ]:
# Train lại trên toàn bộ 100% dữ liệu để mô hình học được nhiều nhất có thể
print("Đang huấn luyện lại trên toàn bộ dữ liệu để xuất file...")
xgb_multi_adv.fit(X_surrogate, y_surrogate)

# Lưu mô hình ra file
joblib.dump(xgb_multi_adv, 'surrogate_model.pkl')
print("Đã lưu thành công file 'surrogate_model.pkl'. Hãy chép file này vào thư mục 'model/' của API.")

Đang huấn luyện lại trên toàn bộ dữ liệu để xuất file...
Đã lưu thành công file 'surrogate_model.pkl'. Hãy chép file này vào thư mục 'model/' của API.


In [ ]:
from google.colab import files
files.download("surrogate_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>